# Issue #28 — Correlación Google Trends vs Volumen de reseñas (semanal)

**Inputs (local):**
- `data/raw/google_trends/*.csv`
- `data/raw/csv/pd_info.csv`
- `data/raw/csv/review_data.csv`

**Outputs (generados):**
- `data/processed/trends_weekly.csv`
- `data/processed/reviews_weekly_volume.csv`
- `data/processed/trends_reviews_weekly_merged.csv`
- `data/processed/trends_reviews_correlations.csv`

**Metodología:**
- Se agrega semanalmente (inicio de semana lunes).
- Se normaliza con min-max (0..1) por keyword.
- Se calcula correlación Pearson y Spearman por keyword.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

corr = pd.read_csv("../data/processed/trends_reviews_correlations.csv")
merged = pd.read_csv("../data/processed/trends_reviews_weekly_merged.csv")
merged["week"] = pd.to_datetime(merged["week"])

display(corr)
merged.head()

## 1) Series temporales normalizadas
Se comparan las curvas semanales de interés (Trends) vs volumen de reseñas.

In [ ]:
for k in merged["trend_keyword"].unique():
    sub = merged[merged["trend_keyword"] == k].sort_values("week")

    plt.figure()
    plt.plot(sub["week"], sub["interest_norm"], label="Trends (norm 0..1)")
    plt.plot(sub["week"], sub["n_reviews_norm"], label="Volumen reseñas (norm 0..1)")
    plt.title(f"Serie semanal - {k}")
    plt.xlabel("Semana")
    plt.ylabel("Valor normalizado")
    plt.legend()
    plt.xticks(rotation=45, ha="right")
    plt.show()

## 2) Scatter y correlación
Se grafica la relación entre interés y volumen para observar tendencia y se reportan Pearson/Spearman.

In [ ]:
for k in merged["trend_keyword"].unique():
    sub = merged[merged["trend_keyword"] == k].dropna(subset=["interest_norm","n_reviews_norm"])

    plt.figure()
    plt.scatter(sub["interest_norm"], sub["n_reviews_norm"])
    plt.title(f"Scatter Trends vs Volumen - {k}")
    plt.xlabel("Trends (norm 0..1)")
    plt.ylabel("Volumen reseñas (norm 0..1)")
    plt.show()

corr

## Hallazgos preliminares
- Para **ácido hialurónico**, la correlación es positiva (Pearson ≈ 0.34, Spearman ≈ 0.41), lo que sugiere asociación moderada entre picos de interés y picos de reseñas semanales.
- Para **niacinamida**, la correlación también es positiva (Pearson ≈ 0.26, Spearman ≈ 0.40).
- No se observó volumen suficiente para **shampoo sin sulfatos** en el dataset de reseñas mapeado (no aparece en `reviews_weekly`), por lo que no se calcula correlación para ese keyword.
- Próximo paso: mejorar el mapeo producto→keyword y ampliar el periodo/datos de reseñas para robustez.